In [ ]:
# -*- coding: utf-8 -*-
import os
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from transformers import pipeline
from google.colab import drive
drive.mount('/content/drive')


MY_DATASET_PATH = "/content/drive/MyDrive/my_ai_research/my_own_dataset"
real_folder = os.path.join(MY_DATASET_PATH, "real")
fake_folder = os.path.join(MY_DATASET_PATH, "fake")

TARGET_SIZE = (224, 224)

def resize_images_in_folder(folder_path):
    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                path = os.path.join(root, f)
                try:
                    img = Image.open(path)
                    if img.size != TARGET_SIZE:
                        img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
                        img_resized.save(path)
                        print(f"Resized: {path}")
                except Exception as e:
                    print(f"Error with {path}: {e}")

print("Проверка и приведение размеров (если нужно)...")
resize_images_in_folder(real_folder)
resize_images_in_folder(fake_folder)
print("Размеры приведены к 224x224.\n")

real_images = [os.path.join(real_folder, f) for f in os.listdir(real_folder) if f.lower().endswith(('.png','.jpg','.jpeg'))]

fake_images = []
for root, dirs, files in os.walk(fake_folder):
    for f in files:
        if f.lower().endswith(('.png','.jpg','.jpeg')):
            fake_images.append(os.path.join(root, f))

print(f"Найдено реальных: {len(real_images)}")
print(f"Найдено сгенерированных: {len(fake_images)}")

all_paths = real_images + fake_images
true_labels = [0]*len(real_images) + [1]*len(fake_images)

models = {
    "AIorNot-SigLIP2": "prithivMLmods/AIorNot-SigLIP2",
    "AI-image-detector": "umm-maybe/AI-image-detector",
    "AI-vs-Deepfake": "prithivMLmods/AI-vs-Deepfake-vs-Real-Siglip2"
}

results = {}

for name, model_id in models.items():
    print(f"\n=== Загрузка модели: {name} ===")
    classifier = pipeline("image-classification", model=model_id)

    preds = []
    for path in tqdm(all_paths, desc=f"Инференс {name}"):
        try:
            res = classifier(path)[0]
            label = res['label'].lower()
            if name == "AI-vs-Deepfake":
                # объединяем ai и deepfake в класс 1 (fake)
                pred = 1 if label in ['ai', 'deepfake'] else 0
            else:
                pred = 1 if label == 'fake' else 0
            preds.append(pred)
        except Exception as e:
            print(f"Ошибка {path}: {e}")
            preds.append(0)   # при ошибке считаем real

    acc = accuracy_score(true_labels, preds)
    report = classification_report(true_labels, preds, target_names=['Real', 'Fake'])
    results[name] = acc
    print(f"Точность {name}: {acc:.2%}")
    print(report)


    result_dir = "/content/drive/MyDrive/my_ai_research/detection_results"
    os.makedirs(result_dir, exist_ok=True)
    with open(os.path.join(result_dir, f"{name}_my_dataset.txt"), "w") as f:
        f.write(f"Model: {name}\n")
        f.write(f"Accuracy: {acc:.4f}\n")
        f.write(report)

print("\n=== ИТОГОВАЯ ТАБЛИЦА ===")
for name, acc in results.items():
    print(f"{name:25} : {acc:.2%}")